# 선박 도장 품질 데이터 라벨링 시각화

불량 이미지의 segmentation 및 bounding box 라벨링을 시각화합니다.


In [14]:
import json
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from pathlib import Path
import random

plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows
# plt.rcParams['font.family'] = 'AppleGothic'  # Mac
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 경로 설정


In [15]:
# 데이터 경로
IMAGE_DIR = 'Sample/01.원천데이터'
LABEL_DIR = 'Sample/02.라벨링데이터'

# 카테고리 정의
CATEGORIES = {
    101: '정상',
    201: '워터스포팅',
    202: '흐름',
    203: '도막분리',
    204: '핀홀',
    205: '균열',
    206: '부풀음',
    207: '이물질포함',
    301: '용접손상',
    302: '스크래치',
    303: '도막떨어짐'
}

# 카테고리별 색상 (BGR)
COLORS = {
    101: (0, 255, 0),      # 정상 - 초록
    201: (255, 0, 0),      # 워터스포팅 - 파랑
    202: (0, 165, 255),    # 흐름 - 주황
    203: (0, 255, 255),    # 도막분리 - 노랑
    204: (255, 0, 255),    # 핀홀 - 마젠타
    205: (0, 0, 255),      # 균열 - 빨강
    206: (255, 255, 0),    # 부풀음 - 시안
    207: (128, 0, 128),    # 이물질포함 - 보라
    301: (255, 165, 0),    # 용접손상 - 하늘
    302: (0, 128, 255),    # 스크래치 - 주황빨강
    303: (128, 128, 128)   # 도막떨어짐 - 회색
}

## 2. 데이터 로드 및 시각화 함수


In [ ]:
def load_json(json_path):
    """JSON 파일 로드"""
    with open(json_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def visualize_annotation(image_path, json_path, show_segmentation=True, show_bbox=True):
    """
    이미지에 어노테이션을 시각화
    
    Args:
        image_path: 이미지 파일 경로
        json_path: JSON 어노테이션 파일 경로
        show_segmentation: segmentation 폴리곤 표시 여부
        show_bbox: bounding box 표시 여부
    """
        # 이미지 경로 확인
    if not os.path.exists(image_path):
        print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")
        return
    
    # 이미지 로드
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ 이미지를 로드할 수 없습니다: {image_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # JSON 로드
    data = load_json(json_path)
    
    # 시각화 준비
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    
    # 원본 이미지
    axes[0].imshow(img)
    axes[0].set_title('원본 이미지', fontsize=16)
    axes[0].axis('off')
    
    # 어노테이션 오버레이 이미지
    overlay = img.copy()
    
    # 각 어노테이션 처리
    for ann in data['annotations']:
        category_id = ann['category_id']
        category_name = CATEGORIES.get(category_id, 'Unknown')
        color = COLORS.get(category_id, (255, 255, 255))
        
        # Segmentation 그리기
        if show_segmentation and 'segmentation' in ann:
            seg = ann['segmentation']
            # 폴리곤 포인트를 reshape
            points = np.array(seg).reshape(-1, 2).astype(np.int32)
            
            # 반투명 폴리곤 오버레이
            overlay_temp = overlay.copy()
            cv2.fillPoly(overlay_temp, [points], color)
            cv2.addWeighted(overlay_temp, 0.4, overlay, 0.6, 0, overlay)
            
            # 폴리곤 경계선
            cv2.polylines(overlay, [points], True, color, 3)
        
        # Bounding Box 그리기
        if show_bbox and 'bbox' in ann:
            x, y, w, h = [int(v) for v in ann['bbox']]
            cv2.rectangle(overlay, (x, y), (x+w, y+h), color, 3)
            
            # 레이블 텍스트
            label = f"{category_name}"
            if 'attributes' in ann and 'part' in ann['attributes']:
                label += f" ({ann['attributes']['part']})"
            
            # 텍스트 배경
            (text_width, text_height), _ = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.8, 2
            )
            cv2.rectangle(overlay, (x, y-text_height-10), 
                         (x+text_width, y), color, -1)
            cv2.putText(overlay, label, (x, y-5), 
                       cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    
    # 어노테이션 오버레이 이미지 표시
    axes[1].imshow(overlay)
    axes[1].set_title('라벨링 시각화', fontsize=16)
    axes[1].axis('off')
    
    # 메타정보 표시
    info_text = f"파일명: {data['images'][0]['file_name']}\n"
    info_text += f"해상도: {data['images'][0]['width']} x {data['images'][0]['height']}\n"
    info_text += f"어노테이션 개수: {len(data['annotations'])}개\n"
    
    for i, ann in enumerate(data['annotations'], 1):
        cat_name = CATEGORIES.get(ann['category_id'], 'Unknown')
        info_text += f"  #{i} {cat_name}"
        if 'area' in ann:
            info_text += f" (면적: {ann['area']:.0f}px²)"
        info_text += "\n"
    
    plt.figtext(0.5, 0.01, info_text, ha='center', fontsize=10, 
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.subplots_adjust(bottom=0.15)
    plt.show()

## 3. 전체 데이터셋 탐색


In [17]:
def get_all_samples():
    """모든 샘플 파일 경로를 수집"""
    samples = []
    
    for category in ['도막 손상', '도장 불량', '양품']:
        label_category_dir = os.path.join(LABEL_DIR, category)
        
        if not os.path.exists(label_category_dir):
            continue
            
        for subcategory in os.listdir(label_category_dir):
            subcategory_dir = os.path.join(label_category_dir, subcategory)
            
            if not os.path.isdir(subcategory_dir):
                continue
            
            for json_file in os.listdir(subcategory_dir):
                if not json_file.endswith('.json'):
                    continue
                
                json_path = os.path.join(subcategory_dir, json_file)
                
                # 이미지 파일 찾기
                image_filename = json_file.replace('.json', '')
                # JPG 또는 jpg 확장자 시도
                image_path_jpg = os.path.join(IMAGE_DIR, category, subcategory, image_filename + '.JPG')
                image_path_jpg_lower = os.path.join(IMAGE_DIR, category, subcategory, image_filename + '.jpg')
                
                if os.path.exists(image_path_jpg):
                    image_path = image_path_jpg
                elif os.path.exists(image_path_jpg_lower):
                    image_path = image_path_jpg_lower
                else:
                    continue
                
                samples.append({
                    'category': category,
                    'subcategory': subcategory,
                    'image_path': image_path,
                    'json_path': json_path
                })
    
    return samples

# 모든 샘플 수집
all_samples = get_all_samples()
print(f"전체 샘플 수: {len(all_samples)}개")

# 카테고리별 통계
from collections import Counter
category_counts = Counter([s['category'] for s in all_samples])
subcategory_counts = Counter([f"{s['category']}/{s['subcategory']}" for s in all_samples])

print("\n=== 카테고리별 분포 ===")
for cat, count in category_counts.items():
    print(f"{cat}: {count}개")

print("\n=== 세부 카테고리별 분포 ===")
for subcat, count in sorted(subcategory_counts.items()):
    print(f"{subcat}: {count}개")

전체 샘플 수: 219개

=== 카테고리별 분포 ===
도막 손상: 54개
도장 불량: 105개
양품: 60개

=== 세부 카테고리별 분포 ===
도막 손상/도막떨어짐: 18개
도막 손상/스크래치: 18개
도막 손상/용접손상: 18개
도장 불량/균열: 15개
도장 불량/도막분리: 15개
도장 불량/부풀음: 15개
도장 불량/워터스포팅: 15개
도장 불량/이물질포함: 15개
도장 불량/핀홀: 15개
도장 불량/흐름: 15개
양품/갑판: 6개
양품/기관실: 6개
양품/선미: 6개
양품/선수: 6개
양품/선실: 6개
양품/엔진커버: 6개
양품/외판: 6개
양품/탱크: 6개
양품/파이프: 6개
양품/해치커버: 6개


## 4. 도막 손상 샘플 시각화


In [18]:
# 도막 손상 샘플 필터링
damage_samples = [s for s in all_samples if s['category'] == '도막 손상']

print(f"도막 손상 샘플: {len(damage_samples)}개")
print("세부 유형:", set([s['subcategory'] for s in damage_samples]))

도막 손상 샘플: 54개
세부 유형: {'용접손상', '스크래치', '도막떨어짐'}


In [ ]:
# 도막떨어짐 샘플 시각화
peeling_samples = [s for s in damage_samples if s['subcategory'] == '도막떨어짐']
sample = random.choice(peeling_samples)

print(f"도막떨어짐 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

도막떨어짐 샘플 시각화


error: OpenCV(4.12.0) D:\bld\libopencv_1766492634538\work\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'


In [ ]:
# 스크래치 샘플 시각화
scratch_samples = [s for s in damage_samples if s['subcategory'] == '스크래치']
sample = random.choice(scratch_samples)

print(f"스크래치 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 용접손상 샘플 시각화
welding_samples = [s for s in damage_samples if s['subcategory'] == '용접손상']
sample = random.choice(welding_samples)

print(f"용접손상 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

## 5. 도장 불량 샘플 시각화


In [ ]:
# 도장 불량 샘플 필터링
defect_samples = [s for s in all_samples if s['category'] == '도장 불량']

print(f"도장 불량 샘플: {len(defect_samples)}개")
print("세부 유형:", set([s['subcategory'] for s in defect_samples]))

In [ ]:
# 균열 샘플 시각화 (여러 annotation이 있는 경우)
crack_samples = [s for s in defect_samples if s['subcategory'] == '균열']
sample = random.choice(crack_samples)

print(f"균열 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 도막분리 샘플 시각화
separation_samples = [s for s in defect_samples if s['subcategory'] == '도막분리']
sample = random.choice(separation_samples)

print(f"도막분리 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 부풀음 샘플 시각화
blister_samples = [s for s in defect_samples if s['subcategory'] == '부풀음']
sample = random.choice(blister_samples)

print(f"부풀음 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 워터스포팅 샘플 시각화
water_samples = [s for s in defect_samples if s['subcategory'] == '워터스포팅']
sample = random.choice(water_samples)

print(f"워터스포팅 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 이물질포함 샘플 시각화
foreign_samples = [s for s in defect_samples if s['subcategory'] == '이물질포함']
sample = random.choice(foreign_samples)

print(f"이물질포함 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 핀홀 샘플 시각화
pinhole_samples = [s for s in defect_samples if s['subcategory'] == '핀홀']
sample = random.choice(pinhole_samples)

print(f"핀홀 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

In [ ]:
# 흐름 샘플 시각화
flow_samples = [s for s in defect_samples if s['subcategory'] == '흐름']
sample = random.choice(flow_samples)

print(f"흐름 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

## 6. 양품 샘플 확인


In [ ]:
# 양품 샘플 (annotation이 없음)
normal_samples = [s for s in all_samples if s['category'] == '양품']
sample = random.choice(normal_samples)

print(f"양품 샘플 시각화")
visualize_annotation(sample['image_path'], sample['json_path'])

## 7. 어노테이션 통계 분석


In [ ]:
# 어노테이션 통계 수집
annotation_stats = []

for sample in all_samples:
    data = load_json(sample['json_path'])
    
    for ann in data['annotations']:
        stat = {
            'category': sample['category'],
            'subcategory': sample['subcategory'],
            'category_id': ann['category_id'],
            'category_name': CATEGORIES.get(ann['category_id'], 'Unknown'),
            'has_segmentation': 'segmentation' in ann,
            'has_bbox': 'bbox' in ann,
            'area': ann.get('area', 0),
            'part': ann.get('attributes', {}).get('part', 'Unknown'),
            'quality': ann.get('attributes', {}).get('quality', 'Unknown')
        }
        annotation_stats.append(stat)

import pandas as pd
df_stats = pd.DataFrame(annotation_stats)

print("=== 어노테이션 통계 ===")
print(f"\n전체 어노테이션 수: {len(df_stats)}개")
print(f"\nSegmentation 있음: {df_stats['has_segmentation'].sum()}개")
print(f"Bounding Box 있음: {df_stats['has_bbox'].sum()}개")

print("\n=== 카테고리별 어노테이션 수 ===")
print(df_stats['category_name'].value_counts())

print("\n=== 품질별 어노테이션 수 ===")
print(df_stats['quality'].value_counts())

print("\n=== 부위별 어노테이션 수 ===")
print(df_stats['part'].value_counts())

In [ ]:
# 불량 영역 면적 분포
defect_areas = df_stats[df_stats['area'] > 0]['area']

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.hist(defect_areas, bins=30, edgecolor='black')
plt.xlabel('면적 (px²)', fontsize=12)
plt.ylabel('빈도', fontsize=12)
plt.title('불량 영역 면적 분포', fontsize=14)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.boxplot(defect_areas)
plt.ylabel('면적 (px²)', fontsize=12)
plt.title('불량 영역 면적 박스플롯', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"평균 불량 영역 면적: {defect_areas.mean():.2f} px²")
print(f"중앙값: {defect_areas.median():.2f} px²")
print(f"최소: {defect_areas.min():.2f} px²")
print(f"최대: {defect_areas.max():.2f} px²")

## 8. 특정 샘플 탐색기


In [ ]:
# 원하는 카테고리와 서브카테고리 선택
selected_category = '도장 불량'  # '도막 손상', '도장 불량', '양품'
selected_subcategory = '균열'  # 위 분포에서 확인한 세부 카테고리

filtered = [s for s in all_samples 
            if s['category'] == selected_category 
            and s['subcategory'] == selected_subcategory]

print(f"{selected_category}/{selected_subcategory} 샘플: {len(filtered)}개")

# 랜덤하게 3개 시각화
for sample in random.sample(filtered, min(3, len(filtered))):
    visualize_annotation(sample['image_path'], sample['json_path'])